# HG4052 · Week 1 Practical
## From zero to a vowel space you built

**No installs, no setup, no experience assumed.**

By the end you will have:
- ✅ a vowel-space plot with IPA labels, axes reversed phonetician-style, made by you
- ✅ every vowel ranked by similarity to /i/, two ways
- ✅ one error message you read and fixed yourself
- ✅ `pwd`, `ls` and `cd` in your fingers

**How this notebook works.** Click a cell, press **Shift + Enter** to run it, and the output appears underneath. Cells marked **✏️ TODO** have one small blank for you to fill (it is always one line or less). Everything else is ready to run. If something breaks, that is not a problem, it is Section 0.


---
## 0 · Orientation: run, break, navigate

Three jobs in this section: run your first cell, break something on purpose, and learn where files live.


In [ ]:
# Run me first: click here, then press Shift + Enter
greeting = "Hello, HG4052!"
print(greeting)
print("Python speaks IPA out of the box: [ə] [ɪ] [ʃ] [ŋ] [ʔ]")

**Micro-task.** This text cell is editable too: double-click it, replace NAME below with your own name, then press Shift + Enter to render it again.

> Notebook of: **NAME**


In [ ]:
# ✏️ This cell is DESIGNED TO FAIL. Run it anyway.
# Then read the error message from the BOTTOM line upward:
#   the last line names the problem, the arrow shows where it happened.
print(greetting)

**What just happened?** Python said `NameError: name 'greetting' is not defined`, because `greetting` (two t's) was never created; `greeting` was. Fix the spelling in the cell above and run it again.

Error messages are information, not verdicts. You will see hundreds of them this term. They do not bite.


### Where am I? Paths and the command line

A **path** is an address, read right to left: `/content/data/vowels.csv` is a file, inside a folder, inside a folder. In Colab, a cell line starting with `!` runs a **shell command** instead of Python:

| command | meaning |
|---|---|
| `pwd` | where am I standing? |
| `ls` | what is here? |
| `cd data` | step into the `data` folder (`cd ..` steps back out) |


In [ ]:
!pwd
!ls

**Get today's data.** One `wget` fetches everything this notebook needs into a `data/` folder. The cell checks its own work and tells you whether to use Plan B.


In [ ]:
!mkdir -p data
!wget -q -O data/vowels.csv https://raw.githubusercontent.com/chenchenzi/hg4052-materials/main/week01/vowels.csv

import os
if os.path.exists("data/vowels.csv") and os.path.getsize("data/vowels.csv") > 100:
    print("✅ Download worked: data/vowels.csv is ready. Skip Plan B and carry on.")
else:
    print("❌ Download failed (the file is missing or empty). Run the Plan B cell below.")

**Plan B (also completely fine).** If the cell above said ❌, run this one instead; it writes the same data directly.


In [ ]:
%%writefile data/vowels.csv
vowel,F1,F2
i,250,2250
i,262,2316
i,270,2276
i,280,2298
i,288,2310
ɪ,370,1950
ɪ,382,2016
ɪ,390,1976
ɪ,400,1998
ɪ,408,2010
ɛ,510,1800
ɛ,522,1866
ɛ,530,1826
ɛ,540,1848
ɛ,548,1860
æ,640,1680
æ,652,1746
æ,660,1706
æ,670,1728
æ,678,1740
ɑ,710,1050
ɑ,722,1116
ɑ,730,1076
ɑ,740,1098
ɑ,748,1110
ɔ,550,800
ɔ,562,866
ɔ,570,826
ɔ,580,848
ɔ,588,860
ʊ,420,980
ʊ,432,1046
ʊ,440,1006
ʊ,450,1028
ʊ,458,1040
u,280,830
u,292,896
u,300,856
u,310,878
u,318,890

In [ ]:
# Where did it go? -l shows sizes, so an empty file cannot hide:
!ls -l data
!head -5 data/vowels.csv

In [ ]:
# One quirk: in a notebook, cd needs a % instead of ! (with !, Python
# forgets the move immediately). Step in, look around, step back out:
%cd data
!pwd
%cd ..

**One more thing to know (not needed today).** Everything in `/content` is wiped when this notebook's runtime shuts down. To keep files, or to use your own recordings later in the course, you can mount your Google Drive; your files then appear under `/content/drive/MyDrive/`. We will need this from Week 3; today, just know it exists.


In [ ]:
# Optional demo, not needed today. Uncomment to try (it will ask for permission):
# from google.colab import drive
# drive.mount('/content/drive')

---
## 1 · Lists and loops over real formant data

The file you just fetched holds 40 vowel tokens: a label plus the first two formants (F1, F2) in Hz, built around the Peterson & Barney (1952) male-speaker means from the lecture.


In [ ]:
import csv

rows = []
with open("data/vowels.csv") as f:
    for r in csv.DictReader(f):
        rows.append((r["vowel"], float(r["F1"]), float(r["F2"])))

print(f"Loaded {len(rows)} vowel tokens")
rows[:5]

In [ ]:
# A loop visits each token in turn. Run me:
for vowel, f1, f2 in rows[:8]:
    print(f"vowel /{vowel}/: F1 = {f1:.0f} Hz, F2 = {f2:.0f} Hz")

In [ ]:
# ✏️ TODO: print only the /u/ tokens.
# Replace the ... with the vowel symbol you want, in quotes: "u"
for vowel, f1, f2 in rows:
    if vowel == ...:
        print(f"vowel /{vowel}/: F1 = {f1:.0f} Hz, F2 = {f2:.0f} Hz")

---
## 2 · numpy: means and difference vectors

numpy turns lists of numbers into **vectors** you can do arithmetic on, exactly the arrows from the lecture. First we group the tokens by vowel and average them: each vowel's **mean vector** is its prototype.


In [ ]:
import numpy as np

tokens = {}
for vowel, f1, f2 in rows:
    tokens.setdefault(vowel, []).append([f1, f2])

means = {v: np.mean(np.array(t), axis=0) for v, t in tokens.items()}

for v, m in means.items():
    print(f"/{v}/   mean F1 = {m[0]:.0f} Hz   mean F2 = {m[1]:.0f} Hz")

# You can also look up one vowel's mean vector directly, by its symbol:
print("\nmeans['i'] =", means["i"])

In [ ]:
# ✏️ TODO: the difference vector from /i/ to /ɪ/, one subtraction.
# Shape of the answer:   diff = means["?"] - means["?"]
# Fill the two vowel symbols (copy-paste ɪ from this line if it is hard to type).
# The lecture says the result should be (+120, -300).
diff = ...
print("difference vector /ɪ/ - /i/ =", diff)

**Read your answer like a phonetician.** F1 went up and F2 went down: the recipe for moving from /i/ to /ɪ/, written as two numbers. (Careful with raw Hz, though: proportionally the F1 change is the bigger one, as the lecture's misconception clinic warned.)


---
## 3 · Plot the vowel space

Now the payoff: the IPA vowel chart, rebuilt from numbers. Phonetician's convention: **F2 on the x-axis, reversed; F1 on the y-axis, reversed**, so front-close vowels sit top-left, like the chart you know.

Run the cell as-is first. Something will look wrong. Then fill the TODO.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5.5))

for v, t in tokens.items():
    arr = np.array(t)
    ax.scatter(arr[:, 1], arr[:, 0], alpha=0.45)          # x = F2, y = F1

for v, m in means.items():
    ax.text(m[1], m[0], v, fontsize=20, ha="center", va="center")

ax.invert_xaxis()                                          # F2 reversed: back vowels to the right
# ✏️ TODO: one more line so vowel HEIGHT runs the right way (open vowels at the bottom).
# It looks exactly like the line above, but for the other axis.


ax.set_xlabel("← F2 (Hz)")
ax.set_ylabel("← F1 (Hz)")
ax.set_title("Peterson & Barney (1952): the vowel chart, rediscovered by measurement")
plt.show()

**Checkpoint.** Compare with the IPA chart: /i/ top-left, /ɑ/ bottom-right-ish, /u/ top-right. If your plot is upside-down, the TODO line is missing; if it mirrors left-right strangely, check which column went on which axis.


---
## 4 · Similarity, two ways

The hardest idea of the week, now executable. Cosine similarity is given; Euclidean distance is the one line the lecture promised you would write yourself.


In [ ]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def distance(a, b):
    # ✏️ TODO: the promised line. Square the gaps, add them up, take the square root.
    # Building blocks, in order:   np.sqrt(   np.sum(   (a - b)**2   )   )
    return ...

# Sanity check against the lecture's worked example (expect about 0.997 and 323):
print("cos(i, ɪ)  =", round(cosine(means["i"], means["ɪ"]), 3))
d = distance(means["i"], means["ɪ"])
print("d(i, ɪ)    =", "⬆ fill the TODO first, then re-run" if d is ... else f"{round(d)} Hz")

In [ ]:
# Rank every vowel by similarity to /i/, both ways:
i = means["i"]
others = [v for v in means if v != "i"]

by_cos  = sorted(others, key=lambda v: cosine(i, means[v]), reverse=True)
by_dist = sorted(others, key=lambda v: distance(i, means[v]))

print("most similar to /i/ ............... least similar")
print("by cosine (angle):   ", "  ".join(by_cos))
print("by distance (gap):   ", "  ".join(by_dist))

**Discuss with your neighbour (2 min).** Where do the two rankings disagree, and why? Remember the lecture: the angle forgives overall size, the distance does not. Which ranking matches your phonetic intuition better, and is that answer the same for every pair?


---
## ✅ Done looks like

- a vowel-space plot with IPA labels, axes flipped
- the two rankings printed above
- the fixed `greeting` cell from Section 0
- `pwd`, `ls` and `cd` in your fingers

Keep this notebook: the skills here return in Assignment 1. If you finished early, the stretch below is yours; otherwise it is a take-home.


---
## 5 · Stretch (or take-home): the vowel guesser

A nearest-neighbour classifier in four lines: given an (F1, F2) point, guess the vowel whose prototype is closest. This is a real, working speech classifier, and it is Week 5's recognizer in miniature.


In [ ]:
def guess_vowel(f1, f2):
    point = np.array([f1, f2])
    return min(means, key=lambda v: distance(point, means[v]))

print(guess_vowel(280, 2250), "  (expected: i)")
print(guess_vowel(700, 1150), "  (expected: ɑ)")

**Try your own voice (optional, as always).** Record a vowel in Praat, measure F1 and F2 at the midpoint (as in HG2003), and feed the numbers to `guess_vowel`. Does a chart built from 1950s American men classify YOUR vowel correctly? If not, you have just met the problem that Week 9 is about.


---
## Before you go

Exit ticket (one sentence, on NTULearn): what is still muddiest from today?

**Next week:** probability and Bayes' rule, the math of a listener's guesses. "Wreck a nice beach" awaits.
